In [ ]:
# 1. SETUP: Install dependencies and check GPU
!pip install -q tensorflow scikit-learn pillow tqdm

import tensorflow as tf
import numpy as np
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))}")
print(f"GPU Devices: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# 2. DATASET SETUP: Mount FaceForensics from Kaggle datasets
import os

kaggle_input = Path('/kaggle/input')
print("Available datasets in /kaggle/input:")
for item in kaggle_input.iterdir():
    print(f"  - {item.name}")

# Note: You need to add FaceForensics dataset to this notebook's data sources

In [ ]:
# 3. DATA PREPARATION: Import dataset utilities

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications.xception import preprocess_input

# Setup paths
WORKING_DIR = Path('/kaggle/working')
OUTPUT_DIR = WORKING_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset directory (assumes FaceForensics is mounted at /kaggle/input)
DATASET_ROOT = Path('/kaggle/input')  # Update this based on mounted dataset name

print(f"Working directory: {WORKING_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# 4. DATA LOADING: Create data generators for training

IMAGE_SIZE = 224
BATCH_SIZE = 32

# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=(0.8, 1.2),
    fill_mode='nearest'
)

# Validation data generator (no augmentation)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

try:
    train_dir = DATASET_ROOT / 'train'
    val_dir = DATASET_ROOT / 'validation'
    
    train_gen = train_datagen.flow_from_directory(
        str(train_dir),
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='binary',
        shuffle=True
    )
    
    val_gen = val_datagen.flow_from_directory(
        str(val_dir),
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='binary',
        shuffle=False
    )
    
    val_eval_gen = val_datagen.flow_from_directory(
        str(val_dir),
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='binary',
        shuffle=False
    )
    
    print(f"✓ Training samples: {len(train_gen.filenames)}")
    print(f"✓ Validation samples: {len(val_gen.filenames)}")
    print(f"✓ Class indices: {train_gen.class_indices}")
    
except Exception as e:
    print(f"✗ Error loading dataset: {e}")
    print("Make sure FaceForensics dataset is added to this notebook's input sources")

In [ ]:
# 5. COMPUTE CLASS WEIGHTS: Handle class imbalance

def compute_class_weights(generator):
    classes = np.asarray(generator.classes)
    total = float(len(classes))
    count_fake = float(np.sum(classes == 0))
    count_real = float(np.sum(classes == 1))
    
    weights = {
        0: total / (2.0 * count_fake) if count_fake > 0 else 1.0,
        1: total / (2.0 * count_real) if count_real > 0 else 1.0,
    }
    return weights

class_weights = compute_class_weights(train_gen)
print(f"Class weights: {class_weights}")

In [ ]:
# 6. BUILD MODEL: Xception with custom head

from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Load pretrained Xception (ImageNet weights)
base_model = Xception(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze backbone for Stage 1
base_model.trainable = False

# Build custom classification head
inputs = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

# Compile for Stage 1
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

print("✓ Model built successfully")
print(f"  Parameters: {model.count_params():,}")

In [ ]:
# 7. STAGE 1 TRAINING: Train classification head only (frozen backbone)

best_weights_path = OUTPUT_DIR / 'xceptionnet_best.h5'

callbacks_stage1 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_weights_path),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("\n" + "="*60)
print("STAGE 1: Training Classification Head (Frozen Backbone)")
print("="*60)

history_stage1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_stage1,
    verbose=1
)

In [ ]:
# 8. STAGE 2: Fine-tune backbone layers

# Unfreeze top 30 layers of backbone
base_model.trainable = True
total_layers = len(base_model.layers)
freeze_until = total_layers - 30

for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

# Freeze BatchNormalization layers during fine-tuning
for layer in base_model.layers[freeze_until:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

trainable_params = sum(p.numpy().size for p in model.trainable_weights)
total_params = model.count_params()
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,}")
print(f"Trainable %: {trainable_params/total_params*100:.1f}%")

In [ ]:
# 9. STAGE 2 TRAINING: Fine-tune backbone

callbacks_stage2 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_weights_path),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc',
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

print("\n" + "="*60)
print("STAGE 2: Fine-tuning Backbone Layers")
print("="*60)

history_stage2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_stage2,
    verbose=1
)

In [ ]:
# 10. LOAD BEST WEIGHTS: Restore best model from training

if best_weights_path.exists():
    model.load_weights(str(best_weights_path))
    print(f"✓ Loaded best validation weights from: {best_weights_path}")

In [ ]:
# 11. THRESHOLD CALIBRATION: Find optimal decision threshold

def find_best_threshold(y_true, probs):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs).ravel()
    best_threshold = 0.5
    best_f1 = -1.0
    
    for threshold in np.linspace(0.1, 0.9, 81):
        preds = (probs >= threshold).astype(int)
        score = f1_score(y_true, preds, average='macro', zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = threshold
    
    return best_threshold

# Get predictions on validation set
val_eval_gen.reset()
probs = model.predict(val_eval_gen, verbose=0).ravel()
threshold_labels = val_eval_gen.classes[:len(probs)]
best_threshold = find_best_threshold(threshold_labels, probs)

print(f"✓ Best threshold for REAL classification: {best_threshold:.4f}")

In [ ]:
# 12. SAVE MODEL: Export to .keras format with metadata

model_name = 'xceptionnet_real_fake'
model_path = OUTPUT_DIR / f'{model_name}.keras'
classes_path = OUTPUT_DIR / f'{model_name}.classes.json'
threshold_path = OUTPUT_DIR / f'{model_name}.threshold.json'

# Save model
model.save(str(model_path))
print(f"✓ Saved model: {model_path}")

# Save class indices
class_indices = train_gen.class_indices
with open(classes_path, 'w') as f:
    json.dump(class_indices, f, indent=2)
print(f"✓ Saved class indices: {classes_path}")

# Save optimal threshold
threshold_data = {'real_threshold': float(best_threshold)}
with open(threshold_path, 'w') as f:
    json.dump(threshold_data, f, indent=2)
print(f"✓ Saved threshold: {threshold_path}")

In [ ]:
# 13. EVALUATION: Comprehensive metrics on validation set

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

val_eval_gen.reset()
probs = model.predict(val_eval_gen, verbose=0).ravel()
preds = (probs >= best_threshold).astype(int)
y_true = val_eval_gen.classes[:len(probs)]

# Classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true, preds, target_names=['FAKE', 'REAL'], digits=4))

# AUC
auc_score = roc_auc_score(y_true, probs)
print(f"\nAUC-ROC Score: {auc_score:.4f}")

# Confusion matrix
cm = confusion_matrix(y_true, preds)
print(f"\nConfusion Matrix:\n{cm}")

In [ ]:
# 14. PLOT RESULTS: Training curves and confusion matrix

# Merge histories from both stages
merged_history = {}
for key in history_stage1.history.keys():
    merged_history[key] = history_stage1.history[key] + history_stage2.history[key]

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy
axes[0, 0].plot(merged_history['accuracy'], label='Train')
axes[0, 0].plot(merged_history['val_accuracy'], label='Validation')
axes[0, 0].set_title('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(merged_history['loss'], label='Train')
axes[0, 1].plot(merged_history['val_loss'], label='Validation')
axes[0, 1].set_title('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# AUC
axes[1, 0].plot(merged_history['auc'], label='Train')
axes[1, 0].plot(merged_history['val_auc'], label='Validation')
axes[1, 0].set_title('AUC')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Confusion Matrix
im = axes[1, 1].imshow(cm, cmap='Blues')
axes[1, 1].set_title('Confusion Matrix')
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_yticks([0, 1])
axes[1, 1].set_xticklabels(['FAKE', 'REAL'])
axes[1, 1].set_yticklabels(['FAKE', 'REAL'])

# Add values to confusion matrix
for i in range(2):
    for j in range(2):
        axes[1, 1].text(j, i, str(cm[i, j]), ha='center', va='center', 
                        color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.colorbar(im, ax=axes[1, 1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_results.png', dpi=150, bbox_inches='tight')
print("✓ Saved training results plot")
plt.show()

In [ ]:
# 15. SUMMARY: Print final metrics

print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)
print(f"\nModel: XceptionNet")
print(f"Dataset: FaceForensics")
print(f"Training samples: {len(train_gen.filenames)}")
print(f"Validation samples: {len(val_gen.filenames)}")
print(f"\nFinal Metrics:")
print(f"  Accuracy: {100 * np.mean(preds == y_true):.2f}%")
print(f"  AUC-ROC:  {auc_score:.4f}")
print(f"  Real Threshold: {best_threshold:.4f}")
print(f"\nOutput Files:")
print(f"  ✓ {model_path.name}")
print(f"  ✓ {classes_path.name}")
print(f"  ✓ {threshold_path.name}")
print(f"  ✓ training_results.png")
print(f"\nAll files saved to: {OUTPUT_DIR}")
print("Ready for download! 🎉")